# Environment: observar y transicionar · 2.1.4

Input 2, 3, 100 → observe/run/update/stop → total=5, dos registros. state es estado del episodio; Memory guarda episodios terminados. No hay reset/step/reward públicos.

In [2]:
import pydantic
import agentic_systems as toolkit
assert toolkit.__version__ == '2.1.4'


In [3]:
"""Episode state and episodic Memory are separate concepts."""
import agentic_systems as toolkit

class State(pydantic.BaseModel):
    amount: int | None = None
    previous: int | None = None
    total: int | None = None

class Output(pydantic.BaseModel):
    total: int

def build():
    graph = toolkit.Graph(State, output_schema=Output)
    def sum_step(state):
        return {'total': state.previous + state.amount}
    graph.add_node('sum', sum_step)
    graph.add_edge(toolkit.START, 'sum')
    graph.add_edge('sum', toolkit.END)
    return toolkit.Environment('ledger', graph=graph, records=({'amount': 2}, {'amount': 3}, {'amount': 100}), state={'total': 0}, memory=toolkit.Memory(capacity=3), observe=lambda row, state: {'amount': row['amount'], 'previous': state['total']}, update=lambda state, row, result: result.data, stop=lambda state, result: state['total'] >= 5)


In [4]:
episode = build()
result = episode.run()
assert result.ok and result.data['summary']['state'] == {'total': 5}
assert result.data['summary']['processed'] == 2
print(result.data['summary'])


{'state': {'total': 5}, 'processed': 2, 'terminated': True, 'truncated': False, 'reason': 'stop'}
